# 06 — Reject inference and the selection effect

The notebook that bounds everything else in this repo.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from src.config import load_config

cfg = load_config()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)

## 1. The approval rate frames every other result

Matching denominators matters, and a first pass got this wrong: comparing the filtered 589,488-row modelling sample against *all* declined applications mixes a filtered numerator with an unfiltered denominator. The correct comparison uses all accepted loans in the window.

| | Count |
|---|---:|
| Accepted 2012–2015, all terms | 844,905 |
| Declined, same window | 5,891,298 |
| **Approval rate** | **12.54%** |

Every model here is fitted on about one eighth of the people who applied. It describes risk ordering **within an already heavily pre-screened population**.

In [ ]:
from src.data_load import load_rejected

rej = load_rejected(cfg['paths']['raw_rejected'], nrows=2_000_000)
rej['app_dt'] = pd.to_datetime(rej['application_date'], errors='coerce')
print(rej.columns.tolist())
rej.head(3)

## 2. The declined population is not a tail of the approved one

Only six fields overlap, and `Risk_Score` is the only risk measure among them (populated on 55% of declined rows). DTI medians are identical at 17.2 on both sides — **the decision was driven overwhelmingly by score**.

In [ ]:
acc = pd.read_parquet('../data/processed/modelling_sample.parquet')
acc_fico = (acc['fico_range_low'] + acc['fico_range_high']) / 2
r = rej[rej['app_dt'].between('2012-01-01','2015-12-31')]

bins = [0,560,600,640,660,680,700,740,900]
pd.DataFrame({
    'approved': pd.cut(acc_fico, bins).value_counts(normalize=True).sort_index().round(4),
    'declined': pd.cut(r['risk_score'].replace(0, np.nan), bins)
                  .value_counts(normalize=True).sort_index().round(4),
})

**63% of declined applicants sit below 660, where the approved book contains literally nobody.** That is a hard cut-off, not a soft preference. The scorecard has never observed an applicant in that range, so scoring one there is extrapolation, not prediction — and no modelling technique changes that.

## 3. A bound, not a correction

With 87.5% of applicants declined, the implied applicant-pool bad rate spans **19% to 54%** across plausible assumptions. The assumption, not the data, drives the answer.

In [ ]:
obs_bad, w = acc['target'].mean(), 5_891_298 / (844_905 + 5_891_298)
pd.DataFrame([{'assumed_declined_bad_rate': a,
               'implied_applicant_pool_bad_rate': round(obs_bad*(1-w) + a*w, 4)}
              for a in [0.20,0.30,0.40,0.50,0.60]])

## 4. Why no correction is reported

A declined applicant has no outcome. Not a missing one — a **non-existent** one. There is no counterfactual to validate against, so every reject-inference method ever published is, on this data, an assumption dressed as an estimate.

Three claims are defensible:

1. The approved book is 12.5% of applicants, so all results are conditional on passing the platform's filter.
2. The declined population is materially riskier on the one shared risk field, and 63% of it lies outside the approved book's support entirely.
3. Any applicant-pool bad rate between roughly 19% and 54% is consistent with what is observable.

Running a standard technique and reporting a single corrected figure would present an assumption as a finding. **Declining to do that is the result.**